# Build your own RAG system

**Human Centered AI: programming week**

Large language models (LLMs) are impressive, but they only know what was in their training data. They know nothing about *your* documents, *your* school or *your* company. **Retrieval-Augmented Generation (RAG)** fixes this: before the LLM answers, we first **look up** the relevant information and **hand it to the LLM** together with the question.

In this notebook you build a small RAG chatbot that answers questions about a (fictional) student handbook.

**What you will practise**
- Reading a text file and splitting it into chunks
- Turning text into numbers (embeddings) with a local model
- Finding the most similar chunks (retrieval)
- Writing a prompt that gives the LLM the right context (augmentation)
- Letting a local LLM write the answer (generation)

**How to work through this notebook**
- Run cells with `Shift + Enter`.
- Cells with `# TODO` are for you: replace every `...` with the correct code.
- Cells without a TODO are already complete. Read them and run them.

## The big picture

```
                 INDEXING (once)
 handbook.txt --> split into chunks --> embed each chunk --> store the vectors

                 ANSWERING (for every question)
 question --> embed question --> find most similar chunks --> build prompt --> LLM --> answer
              \______________ RETRIEVE ______________/        AUGMENT       GENERATE
```

The three words in the name map directly to the three steps: **R**etrieve, **A**ugment, **G**enerate.

## Step 0: Setup

We use [Ollama](https://ollama.com) to run models locally. Two kinds of models are needed:

| Role | What it does | Model in this notebook |
|---|---|---|
| **Embedding model** | turns text into a list of numbers | `nomic-embed-text` |
| **Language model (LLM)** | writes the answer | `llama3.2` (use any chat model you have) |

Open a terminal and run this once to download the embedding model:

```
ollama pull nomic-embed-text
```

You can see which models you have with `ollama list`.

In [ ]:
%pip install -q ollama numpy

In [ ]:
import numpy as np
import ollama

LLM_MODEL = "llama3.1"            # <- change this to a chat model you have in Ollama
EMBED_MODEL = "nomic-embed-text"   # the model that turns text into numbers

Let's check that Ollama is running and that the LLM answers:

In [ ]:
print(ollama.list())   # shows all models you have downloaded

response = ollama.chat(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
)
print("\n" + response["message"]["content"] + "\n")

## Step 1: The problem, an LLM that doesn't know our data

Hogeschool Rivierstad is a fictional school. Our handbook contains facts about it. First let's see what the LLM does when we ask a question about it **without** any help.

Look at how `ollama.chat` works: you give it a `model` and a list of `messages`. Each message has a `role` (`"user"` is you) and the `content` (the text).

In [ ]:
question = "How many free print credits does a student get per year at Hogeschool Rivierstad?"

response = ollama.chat(
    model=...,                                # TODO: which variable holds the LLM model name?
    messages=[
        {"role": "user", "content": ...},     # TODO: put the question here
    ],
)
print(response["message"]["content"])

**Discuss:** Is the answer correct? Does the model *say* that it doesn't know? This is called a **hallucination**: the model produces a confident, fluent answer that is made up. RAG is one of the most common ways to reduce this.

## Step 2: Load the data

Our knowledge source is a plain text file, `handbook.txt`. In Python you read a file like this:

```python
with open("file.txt", encoding="utf-8") as f:
    text = f.read()
```

`with open(...) as f` opens the file and closes it again automatically. `f.read()` returns the whole content as one big string.

In [ ]:
with open("handbook.txt", encoding="utf-8") as f:
    text = ...                      # TODO: read the whole file

print(text[:500])                   # the first 500 characters
print("...")
print(f"Total number of characters: {len(text)}")

## Step 3: Chunking

We don't give the whole handbook to the LLM for every question. That would be slow, expensive and the model might get distracted. Instead we cut the text into small pieces, called **chunks**, and later only pick the relevant ones.

In our handbook, every topic is a section separated by an empty line. In Python, an empty line between paragraphs is the text `"\n\n"`. The string method `.split()` cuts a string at every occurrence of a separator:

```python
"a,b,c".split(",")     # gives ["a", "b", "c"]
```

After splitting, we clean the result. This line uses a **list comprehension**, a short way to write a loop:

```python
chunks = [c.strip() for c in raw_chunks if c.strip() != ""]
```

which does the same as:

```python
chunks = []
for c in raw_chunks:
    if c.strip() != "":       # skip empty pieces
        chunks.append(c.strip())   # strip() removes spaces/newlines at the edges
```

In [ ]:
raw_chunks = text.split(...)        # TODO: what text separates two sections? (see above)
chunks = [c.strip() for c in raw_chunks if c.strip() != ""]

print(f"Number of chunks: {len(chunks)}")
print()
print("Example chunk:")
print(chunks[3])

## Step 4: Embeddings, turning text into numbers

A computer cannot compare the *meaning* of two texts directly. An **embedding model** solves this: it converts a text into a list of numbers (a **vector**). Texts with a similar meaning get similar vectors.

Let's look at one:

In [ ]:
response = ollama.embed(model=EMBED_MODEL, input="Hello, world!")
vector = response["embeddings"][0]

print(f"Length of the vector: {len(vector)}")
print(f"First 5 numbers: {vector[:5]}")

Now we write a **function** that we can reuse. A function is a named block of code that takes input and gives back (`return`) a result.

`ollama.embed` can take a *list* of texts at once, so our function will do that too. We wrap the result in a NumPy array (`np.array`), which makes the maths on the next steps easy.

In [ ]:
def embed(texts):
    """Turn a list of texts into a NumPy array with one vector per text."""
    response = ollama.embed(model=..., input=...)    # TODO: embedding model name + the texts
    return np.array(response["embeddings"])

### Measuring similarity

How do we know if two vectors are "close"? We use **cosine similarity**: it measures the angle between two vectors.
- close to **1**: very similar meaning
- close to **0**: unrelated
- (negative: opposite)

The formula is: dot product of `a` and `b`, divided by the length of `a` times the length of `b`. NumPy has `np.dot(a, b)` for the dot product and `np.linalg.norm(a)` for the length of a vector.

In [ ]:
def cosine_similarity(a, b):
    """Return a number between -1 and 1. Higher means more similar."""
    dot_product = np.dot(a, b)
    length_a = np.linalg.norm(a)
    length_b = ...                  # TODO: the length of vector b
    return dot_product / (length_a * length_b)

Let's test our two functions. Which of the sentences do you expect to be most similar?

In [ ]:
sentences = [
    "The cat sits on the sofa.",
    "A kitten is lying on the couch.",
    "I need to file my tax return.",
]
vectors = embed(sentences)

print("cat vs kitten:", round(cosine_similarity(vectors[0], vectors[1]), 3))
print("cat vs tax   :", round(cosine_similarity(vectors[0], vectors[2]), 3))

**Try it yourself:** make your own list of 3 sentences in the cell below. Predict the similarities *before* you run it. Can you find two sentences that share many words but have a different meaning? Or two sentences that share *no* words but mean the same?

In [ ]:
# Your own experiments here

## Step 5: Build the index

Now we embed **all** chunks, once. The result is our "vector database". For a small project like this, a NumPy array is all we need: one row per chunk.

(Big projects use dedicated vector databases such as ChromaDB, Qdrant or pgvector, but the idea is exactly the same.)

In [ ]:
chunk_embeddings = ...             # TODO: embed all the chunks with our embed() function

print(f"Shape: {chunk_embeddings.shape}")   # (number of chunks, length of each vector)

## Step 6: Retrieve, find the best chunks for a question

This is the **R** in RAG. The recipe:
1. Embed the question with the *same* embedding model.
2. Compute the cosine similarity between the question and every chunk.
3. Return the `k` chunks with the highest score.

For step 3 we use `np.argsort(scores)`. It does not sort the scores themselves, but gives the **positions (indices)** in sorted order from low to high:

```python
scores = [0.2, 0.9, 0.5]
np.argsort(scores)             # [0, 2, 1]  (position 0 is the lowest, position 1 the highest)
np.argsort(scores)[::-1]       # [1, 2, 0]  (reversed: highest first)
np.argsort(scores)[::-1][:2]   # [1, 2]     (the top 2)
```

In [ ]:
def retrieve(question, k=3):
    """Find the k chunks that are most similar to the question."""
    question_vector = embed([question])[0]

    scores = []
    for chunk_vector in chunk_embeddings:
        score = ...               # TODO: cosine similarity between question_vector and chunk_vector
        scores.append(score)

    best_indices = np.argsort(scores)[::-1][:k]   # positions of the top k, highest first

    return [(scores[i], chunks[i]) for i in best_indices]

Test it. For each result we print the score and the title (first line) of the chunk:

In [ ]:
for score, chunk in retrieve("When is the AI lab open?"):
    print(f"{score:.3f}  {chunk.splitlines()[0]}")

**Check:** is the best result the one you expected? Try a few more questions in the cell below. Can you find a question where the wrong chunk comes first?

In [ ]:
for score, chunk in retrieve("Can I get more time during my exam?"):
    print(f"{score:.3f}  {chunk.splitlines()[0]}")

## Step 7: Augment, build the prompt

This is the **A** in RAG. We build one text (the **prompt**) that contains:
1. instructions for the LLM,
2. the retrieved chunks as **context**,
3. the user's **question**.

Good instructions matter. Tell the model to use *only* the context, and what to say when the answer is not in it. This keeps it from making things up.

In Python an **f-string** (`f"..."`) lets you put variables inside text using curly braces: `f"Hello {name}"`. With triple quotes (`f"""..."""`) the text can span multiple lines.

In [ ]:
def build_prompt(question, context_chunks):
    """Combine instructions, context and question into one prompt."""
    context = "\n\n".join(context_chunks)    # glue the chunks together with an empty line between

    prompt = f"""
    ...  TODO: write the instructions here. Tell the LLM that it is an assistant for students,
    that it must answer ONLY with the context below, and that it must say
    "I don't know based on the handbook." if the answer is not in the context.

CONTEXT:
{...}

QUESTION:
{...}

ANSWER:"""
    return prompt

## Step 8: Generate, let the LLM answer

The **G** in RAG. This looks a lot like Step 1, but now the prompt contains the context.

We also set `temperature` to 0. Temperature controls randomness: at 0 the model always picks its most likely answer, which is what we want for factual questions.

In [ ]:
def generate_answer(prompt):
    """Send the prompt to the LLM and return its answer as text."""
    response = ollama.chat(
        model=...,                                         # TODO
        messages=[{"role": "user", "content": ...}],       # TODO
        options={"temperature": 0},
    )
    return response["message"]["content"]

## Step 9: Put it all together

Now we connect the three steps into one function, `ask()`.

In [ ]:
def ask(question, k=3, show_sources=False):
    """Answer a question using RAG."""
    # 1. Retrieve: find the k most relevant chunks
    results = ...                                      # TODO: use retrieve()
    context_chunks = [chunk for score, chunk in results]

    # 2. Augment: build the prompt
    prompt = ...                                       # TODO: use build_prompt()

    # 3. Generate: let the LLM answer
    answer = ...                                       # TODO: use generate_answer()

    if show_sources:
        print("Sources used:")
        for score, chunk in results:
            print(f"  - {chunk.splitlines()[0]}  (score {score:.3f})")
        print()

    return answer

Now ask the **same question as in Step 1**. Is the answer different this time?

In [ ]:
print(ask("How many free print credits does a student get per year at Hogeschool Rivierstad?", show_sources=True))

## Step 10: Experiments

A RAG system has many knobs. Let's explore a few. Run the cell below and check each answer against `handbook.txt`.

In [ ]:
questions = [
    "How many weeks does the internship last?",
    "What should I do if I am too sick to take an exam?",
    "Am I allowed to use ChatGPT for my assignments?",
    "What happens if I hand in an assignment 30 hours late?",
    "When is the AI lab open?",
]

for q in questions:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 60)

### Experiment A: a question that is *not* in the handbook

What should a good RAG system do here? Run it and see what ours does.

In [ ]:
print(ask("How much does a cup of coffee cost in the school canteen?", show_sources=True))

### Experiment B: the effect of `k`

How many chunks should we give the LLM? Let's try `k = 1`, `3` and `5` for the same question. Look at the sources and at the answers.

In [ ]:
question = "What happens if I miss the exam because I am sick?"

for k in [...]:                                   # TODO: a list with the values 1, 3 and 5
    print(f"--- k = {k} ---")
    print(...)                                    # TODO: ask the question with this k, show_sources=True
    print()

**Discuss:**
- With `k=1`, what can go wrong? (This question touches several sections: sick reports *and* resits.)
- With a large `k`, what are the downsides? (Think about cost, speed and distraction.)

### Experiment C: improve the prompt

Change the instructions in `build_prompt` (for example: answer in one sentence, answer in Dutch, or include the name of the section you used). Run `ask()` again. How much influence does the wording have?

In [ ]:
# Re-run build_prompt with your own instructions, then ask a question

### Bonus: a Dutch question

Our handbook is in English. Ask a question in Dutch and see what happens. Does retrieval still find the right chunk? (`nomic-embed-text` is mostly trained on English; a multilingual embedding model such as `bge-m3` may do better.)

In [ ]:
print(ask("Hoeveel weken duurt de stage?", show_sources=True))

## Reflection questions

Discuss in pairs, then write down your answer:

1. Why do we split the handbook into chunks instead of embedding the whole text as one vector?
2. What could go wrong if chunks are **too small**? What if they are **too large**?
3. Why did we tell the LLM to say "I don't know" when the answer is not in the context?
4. Name one way the **retrieval** step can fail. How would you notice it as a developer?
5. **Human-centered:** a student asks this bot when to report sick, and the bot gives a wrong answer, so the student misses a deadline. Who is responsible? What could you change in the *interface* (not the model) to help users judge whether they can trust an answer?

## Taking it further (optional)

- Use your own data: a few PDFs, your course notes, the Wikipedia page of your favourite hobby.
- Chunk differently: split by sentence, or use chunks of 500 characters with an overlap of 100. Does the quality change?
- Add chat history so you can ask follow-up questions ("And what about the resit?").
- Replace the NumPy array with a vector database such as ChromaDB.